[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Pandas, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)

# loc and iloc &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import math
from datetime import datetime, timedelta

import pandas as pd

STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}


def year_of_readings():
    """Every hour of 2025 at the four stations, as the sqlite3 guide makes them, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


readings = pd.DataFrame(list(year_of_readings()), columns=["station", "time", "celsius"])
readings["time"] = pd.to_datetime(readings["time"], format="%Y-%m-%dT%H:%M")
stations = pd.DataFrame({"station": list(LATITUDES), "latitude": list(LATITUDES.values())})

print("pandas", pd.__version__, "|", len(readings), "readings,", len(stations), "stations")

pandas 2.2.3 | 35040 readings, 5 stations


**1.** Print the reading in row 100 of `readings`, by label with `loc` and by position with `iloc`, and check that they agree.

In [2]:
by_label = readings.loc[100, "celsius"]
by_position = readings.iloc[100, 2]

print(by_label, by_position, by_label == by_position)

-3.4 -3.4 True


The table has not been sorted, so the row labeled 100 is also the 101st row, and both find the same reading.

**2.** Sort `readings` from warmest to coldest with `sort_values(["celsius", "time"], ascending=[False, True])`, and print the warmest row with `iloc`.

In [3]:
warmest_first = readings.sort_values(["celsius", "time"], ascending=[False, True])

warmest_first.iloc[0]

station                 Bergen
time       2025-07-21 15:00:00
celsius                   20.8
Name: 19356, dtype: object

`iloc[0]` is the first row of the sorted table, Bergen's 20.8 degrees at 15:00 on 21 July. Its label, 19356, shows where it was before the sort.

**3.** Show the last three rows of `readings` with `iloc` and a negative slice.

In [4]:
readings.iloc[-3:]

,station,time,celsius
35037,Oslo,2025-12-31 23:00:00,-3.3
35038,Svalbard,2025-12-31 23:00:00,-14.4
35039,Tromso,2025-12-31 23:00:00,-6.5


`-3:` starts three rows from the end and runs to the end.

**4.** Show the `station` and `celsius` columns for the readings below -17 degrees, with `loc` and a mask.

In [5]:
readings.loc[readings["celsius"] < -17, ["station", "celsius"]]

,station,celsius
594,Svalbard,-17.1
686,Svalbard,-17.1
1070,Svalbard,-17.3
1162,Svalbard,-17.1
1546,Svalbard,-17.2
1842,Svalbard,-17.1
1934,Svalbard,-17.1
2226,Svalbard,-17.1
2318,Svalbard,-17.1
2702,Svalbard,-17.1


Ten readings, all of them Svalbard's, the coldest station, and all in January.

**5.** Make a table indexed by time, as `by_time` is, and show the readings at 15:00 on 21 July 2025.

In [6]:
by_time = readings.set_index("time")

by_time.loc["2025-07-21 15:00"]

,station,celsius
time,,
2025-07-21 15:00:00,Bergen,20.8
2025-07-21 15:00:00,Oslo,19.2
2025-07-21 15:00:00,Svalbard,8.1
2025-07-21 15:00:00,Tromso,16.0


Four rows share that time, one for each station. Bergen's 20.8 is its warmest reading of the year.

---

&#8592; **Back to:** [loc and iloc](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/07-loc-and-iloc.ipynb)  &nbsp;&middot;&nbsp;  [Pandas, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)
